# Local Windows LoRA run

Run this notebook with a native Windows Python kernel from this repository. The environment must already contain the project dependencies, `git` must be on `PATH`, and an NVIDIA CUDA GPU is required. Do not run these cells in WSL.

The notebook stores adapters under `local_artifacts/`, results under `local_results/`, and the external RAG clone under `dms-rag/`. These folders are ignored by Git.

In [1]:
import os
import sys
from pathlib import Path

# Set deterministic CUDA configuration before importing torch or the workflow.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "jura_hypersumm").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from inside the JURA_hypersumm repository.")

REPO_ROOT = find_repository_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
TRAIN_PATH = REPO_ROOT / "train_ternary.csv"
VAL_PATH = REPO_ROOT / "val_ternary.csv"
RAG_DIR = REPO_ROOT / "dms-rag"
ARTIFACT_ROOT = REPO_ROOT / "local_artifacts"
RESULTS_DIR = REPO_ROOT / "local_results"
DOCX_DIR = REPO_ROOT / "local_docx"
DOCX_DIR.mkdir(parents=True, exist_ok=True)

# Put private test decisions in local_docx/. They are read in place and are not deleted.
DOCUMENT_PATHS = sorted(DOCX_DIR.glob("*.docx"))

# Change to True after a completed run to reuse the saved adapter without training.
USE_EXISTING_MODEL = False

print(f"Repository: {REPO_ROOT}")
print(f"DOCX files found: {len(DOCUMENT_PATHS)}")

Repository: D:\CODE\Python\Projects\JURA_hypersumm
DOCX files found: 35


Ministral normally does not require a Hugging Face token. For a gated model such as Llama, set `HF_TOKEN` in the Windows environment before starting Jupyter; do not paste a token into this notebook. If `DOCUMENT_PATHS` is empty, training and validation run normally and document testing is skipped.

In [2]:
from jura_hypersumm.lora import run as _run_lora

def run(model_name, task, hyperparameters=None, **overrides):
    """Run the repository LoRA workflow with native Windows paths."""
    local_options = {
        "train_path": TRAIN_PATH,
        "val_path": VAL_PATH,
        "rag_dir": RAG_DIR,
        "drive_root": ARTIFACT_ROOT,
        "results_dir": RESULTS_DIR,
        "document_paths": DOCUMENT_PATHS,
        "use_existing_model": USE_EXISTING_MODEL,
    }
    local_options.update(overrides)
    return _run_lora(model_name, task, hyperparameters, **local_options)

In [3]:
scores = run("ministral", "ternary", use_existing_model=True)
scores

[JURA][lora/ministral/ternary][SETUP] Preparing datasets and artifact paths.
[JURA][lora/ministral/ternary][REUSE] Checking the previously trained adapter at D:\CODE\Python\Projects\JURA_hypersumm\local_artifacts\models\lora\mistralai_Ministral-8B-Instruct-2410\ternary.
[JURA][lora/ministral/ternary][RAG] Preparing the pinned RAG repository.
[JURA][lora/ministral/ternary][RAG] RAG repository ready at e6eab944161e.
[JURA][lora/ministral/ternary][LOAD] Loading the saved LoRA adapter; training is skipped.


No prebuilt binary for CUDA 12.9, loading CUDA 12.8 instead. Set BNB_CUDA_VERSION to override.
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[JURA][lora/ministral/ternary][LOAD] Saved LoRA adapter loaded.
[JURA][lora/ministral/ternary][VALIDATION] Starting validation on 336 examples.


Validating LoRA ternary:   0%|          | 0/336 [00:00<?, ?batch/s]

...

Урунов ч 3.1 ст. 18.8_Готово.docx [ternary]:   0%|          | 0/5 [00:00<?, ?it/s]

[JURA][lora/ministral/ternary][TESTING] Full document inference finished.
[JURA][lora/ministral/ternary][RESULTS] Writing score and review artifacts.


,model,task,support,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,contradiction_precision,contradiction_recall,contradiction_f1,invalid_predictions
0,mistralai/Ministral-8B-Instruct-2410,ternary,336,0.985119,0.985405,0.985119,0.98509,0.98509,0.973913,1.0,0.986784,0


[JURA][results][LOCAL] Saved artifact: D:\CODE\Python\Projects\JURA_hypersumm\local_results\lora_mistralai_Ministral-8B-Instruct-2410_ternary_20260804T155842Z.xlsx
[JURA][results][LOCAL] Saved artifact: D:\CODE\Python\Projects\JURA_hypersumm\local_results\lora_mistralai_Ministral-8B-Instruct-2410_ternary_document_review_20260804T155843Z.zip
[JURA][lora/ministral/ternary][COMPLETE] Workflow finished.


,model,task,support,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,contradiction_precision,contradiction_recall,contradiction_f1,invalid_predictions
0,mistralai/Ministral-8B-Instruct-2410,ternary,336,0.985119,0.985405,0.985119,0.98509,0.98509,0.973913,1.0,0.986784,0
